# EloSense v2: PGN Move-Level Features

Does move-level game data (opening, length, material swings) predict skill better than metadata alone? The v1 notebook found metadata-only accuracy drops to 39% without the opponent's rating band. This notebook tests whether real move data closes that gap.

## 1. Setup and PGN Sample

In [ ]:
import io

import chess
import chess.pgn
import pandas as pd

DATA_PATH = "../data/club_games_data.csv"
SAMPLE_SIZE = 3000

df_full = pd.read_csv(DATA_PATH)
sample = df_full.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
sample.shape

In [ ]:
# Proof of concept: parse a single PGN and pull basic info out of it
game = chess.pgn.read_game(io.StringIO(sample["pgn"].iloc[0]))

print("ECO:", game.headers.get("ECO"))
print("White:", game.headers.get("White"), "Black:", game.headers.get("Black"))
print("Result:", game.headers.get("Result"))

moves = list(game.mainline_moves())
print("ply count:", len(moves))
print("first 10 moves:", moves[:10])

## 2. PGN Parsing Function

In [ ]:
def parse_pgn(pgn_text):
    if not isinstance(pgn_text, str):
        return None
    try:
        return chess.pgn.read_game(io.StringIO(pgn_text))
    except Exception as e:
        print(f"Failed to parse PGN: {e}")
        return None

sample["game"] = sample["pgn"].apply(parse_pgn)
fail_count = sample["game"].isna().sum()
print(f"{fail_count} of {len(sample)} PGNs failed to parse")

0 failures on this 3,000-game sample. `parse_pgn` still guards against non-string values and malformed PGNs (logs and returns `None` instead of crashing), since the full 60K+ dataset may hit edge cases this sample didn't.

## 3. Opening Extraction

In [ ]:
def get_eco(game):
    if game is None:
        return None
    return game.headers.get("ECO")

def get_opening_moves(game, n=10):
    if game is None:
        return None
    board = game.board()
    san_moves = []
    for move in list(game.mainline_moves())[:n]:
        san_moves.append(board.san(move))
        board.push(move)
    return " ".join(san_moves)

sample["eco"] = sample["game"].apply(get_eco)
sample["opening_moves"] = sample["game"].apply(get_opening_moves)

print("missing ECO:", sample["eco"].isna().sum(), "of", len(sample))
sample[["eco", "opening_moves"]].head()

ECO code is missing on 36 of 3,000 games (1.2%), those PGNs just don't have an `[ECO]` header. Using ECO as the categorical opening feature since it's already a standard classification, `opening_moves` (first 10 SAN moves) is kept as a readable backup and for sanity-checking games by eye later.